In [13]:
# ============================================================
# 1. Environment Setup
# ============================================================
# Philosophy:
#
# Reproducible machine learning systems require
# consistent library versions.
#
# This project uses SageMaker Pipelines, Model
# Registry, Batch Transform, and CloudWatch.
#
# Installing a compatible SageMaker SDK ensures
# that all pipeline components behave consistently
# across different environments.
#
# If packages are updated, restart the kernel
# before continuing.
# ============================================================

%pip install -U "sagemaker<3.0.0"

Note: you may need to restart the kernel to use updated packages.


In [13]:
# ============================================================
# 2. Orchestrator Notebook Setup
# ============================================================

import os
import sys
import shutil
import importlib

PROJECT_ROOT = os.getcwd()

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

pycache_path = os.path.join("src", "__pycache__")

if os.path.exists(pycache_path):
    shutil.rmtree(pycache_path)

for module_name in list(sys.modules.keys()):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]

importlib.invalidate_caches()

import json
import time
import boto3
import sagemaker
import pandas as pd
import numpy as np

from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.parameters import ParameterInteger, ParameterString, ParameterFloat
from sagemaker.processing import ProcessingInput, ProcessingOutput, ScriptProcessor
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.workflow.steps import ProcessingStep, TrainingStep, TransformStep
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet, Join
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput, TransformInput
from sagemaker.model import Model
from sagemaker.model_metrics import MetricsSource, ModelMetrics
from sagemaker.transformer import Transformer
from sagemaker.s3 import S3Downloader

from src.config import (
    RAW_FILE,
    TARGET,
    SAMPLE_SIZE,
    PRODUCTION_SIZE,
    PIPELINE_NAME,
    MODEL_PACKAGE_GROUP_NAME,
    FEATURE_GROUP_NAME,
    ATHENA_DATABASE,
    ATHENA_TABLE,
    CLOUDWATCH_NAMESPACE,
    DASHBOARD_NAME
)

from src.utils import create_or_reuse_bucket
from src.data_ingestion import upload_raw_dataset
from src.athena_setup import create_athena_database_and_table, run_athena_eda_query
from src.monitoring import publish_pipeline_metrics, create_pipeline_dashboard

sagemaker_session = sagemaker.Session()
pipeline_session = PipelineSession()

REGION = sagemaker_session.boto_region_name
role = sagemaker.get_execution_role()

account_id = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]

BUCKET_NAME = f"aai540-diabetes-readmission-{account_id}"

s3 = boto3.client("s3", region_name=REGION)

create_or_reuse_bucket(BUCKET_NAME, REGION)

print("SageMaker SDK version:", sagemaker.__version__)
print("Region:", REGION)
print("Role:", role)
print("Bucket:", BUCKET_NAME)
print("Pipeline:", PIPELINE_NAME)
print("Model Package Group:", MODEL_PACKAGE_GROUP_NAME)

Bucket already exists:
aai540-diabetes-readmission-468962265940
SageMaker SDK version: 2.257.3
Region: us-east-1
Role: arn:aws:iam::468962265940:role/LabRole
Bucket: aai540-diabetes-readmission-468962265940
Pipeline: DiabetesReadmissionPipeline
Model Package Group: DiabetesReadmissionModelPackageGroup


In [14]:
# ============================================================
# 3. Define Raw Data S3 URI
# ============================================================
# Philosophy:
#
# The EDA notebook uploads the raw diabetes dataset
# to the S3 data lake.
#
# The orchestrator notebook does not recreate
# Athena or Feature Store resources. It only points
# the SageMaker Pipeline to the raw data location
# that already exists in S3.
# ============================================================

raw_data_s3_uri = (
    f"s3://{BUCKET_NAME}/raw/diabetes/diabetic_data.csv"
)

print("Raw data S3 URI used by pipeline:")
print(raw_data_s3_uri)

Raw data S3 URI used by pipeline:
s3://aai540-diabetes-readmission-468962265940/raw/diabetes/diabetic_data.csv


In [15]:
# ============================================================
# 4. Define Pipeline Parameters
# ============================================================
# Philosophy:
#
# Pipeline parameters make the SageMaker Pipeline
# reusable and configurable.
#
# Instead of changing the pipeline code each time,
# we can adjust values at execution time, such as:
#
# - instance types
# - model approval status
# - F1 threshold
# - prediction threshold
# - sample size
# - production batch size
#
# This supports CI/CD because the same pipeline
# definition can be tested under different
# conditions.
# ============================================================

processing_instance_count = ParameterInteger(
    name="ProcessingInstanceCount",
    default_value=1
)

processing_instance_type = ParameterString(
    name="ProcessingInstanceType",
    default_value="ml.m5.large"
)

training_instance_type = ParameterString(
    name="TrainingInstanceType",
    default_value="ml.m5.large"
)

transform_instance_type = ParameterString(
    name="TransformInstanceType",
    default_value="ml.m5.large"
)

model_approval_status = ParameterString(
    name="ModelApprovalStatus",
    default_value="PendingManualApproval"
)

input_data = ParameterString(
    name="InputData",
    default_value=raw_data_s3_uri
)

f1_threshold = ParameterFloat(
    name="F1Threshold",
    default_value=0.20
)

prediction_threshold = ParameterFloat(
    name="PredictionThreshold",
    default_value=0.50
)

sample_size_param = ParameterInteger(
    name="SampleSize",
    default_value=SAMPLE_SIZE
)

production_size_param = ParameterInteger(
    name="ProductionSize",
    default_value=PRODUCTION_SIZE
)

print("Pipeline parameters defined.")
print("Default F1 threshold:", 0.20)
print("Default prediction threshold:", 0.50)

Pipeline parameters defined.
Default F1 threshold: 0.2
Default prediction threshold: 0.5


In [16]:
# ============================================================
# 5. Define Preprocessing Step
# ============================================================
# Philosophy:
#
# The preprocessing step is the first automated
# step inside the SageMaker Pipeline.
#
# It runs src/preprocessing.py as a managed
# SageMaker Processing Job.
#
# This step produces:
#
# - train.csv
# - validation.csv
# - test.csv
# - batch_input.csv
# - production_labeled.csv
# - schema.json
#
# These outputs feed the later training,
# evaluation, Batch Transform, and monitoring steps.
# ============================================================

sklearn_processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type=processing_instance_type,
    instance_count=processing_instance_count,
    base_job_name="diabetes-preprocess",
    sagemaker_session=pipeline_session
)

processor_args = sklearn_processor.run(
    inputs=[
        ProcessingInput(
            source=input_data,
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/train"
        ),
        ProcessingOutput(
            output_name="validation",
            source="/opt/ml/processing/validation"
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/test"
        ),
        ProcessingOutput(
            output_name="batch",
            source="/opt/ml/processing/batch"
        ),
        ProcessingOutput(
            output_name="monitoring",
            source="/opt/ml/processing/monitoring"
        )
    ],
    code="src/preprocessing.py",
    arguments=[
        "--sample-size", sample_size_param.to_string(),
        "--production-size", production_size_param.to_string()
    ]
)

step_process = ProcessingStep(
    name="DiabetesPreprocess",
    step_args=processor_args
)

print("Preprocessing step defined.")

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


Preprocessing step defined.


In [17]:
# ============================================================
# 6. Define Training Step
# ============================================================
# Philosophy:
#
# The training step trains the  champion-candidate
# XGBoost model using src/train.py.
#
# Unlike V1, where hyperparameters were mostly
# defined directly in the notebook,  moves the
# training logic into a separate script.
#
# This is more professional because a data scientist
# can improve train.py without changing the
# orchestration logic.
# ============================================================

xgboost_image_uri = sagemaker.image_uris.retrieve(
    framework="xgboost",
    region=REGION,
    version="1.7-1",
    py_version="py3",
    instance_type="ml.m5.large"
)

model_output_path = f"s3://{BUCKET_NAME}/pipeline/model-artifacts"

xgb_estimator = Estimator(
    image_uri=xgboost_image_uri,
    role=role,
    instance_count=1,
    instance_type=training_instance_type,
    output_path=model_output_path,
    entry_point="train.py",
    source_dir="src",
    sagemaker_session=pipeline_session
)

xgb_estimator.set_hyperparameters(
    num_round=500,
    max_depth=5,
    eta=0.05,
    subsample=0.90,
    colsample_bytree=0.90,
    min_child_weight=3,
    gamma=0.5,
    lambda_reg=1
)

train_args = xgb_estimator.fit(
    inputs={
        "train": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="text/csv"
        ),
        "validation": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            content_type="text/csv"
        )
    }
)

step_train = TrainingStep(
    name="DiabetesTrainXGBoost",
    step_args=train_args
)

print("Training step defined.")

INFO:sagemaker.image_uris:Ignoring unnecessary Python version: py3.


INFO:sagemaker.image_uris:Ignoring unnecessary instance type: ml.m5.large.


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Training step defined.


In [18]:
# ============================================================
# 7. Define Evaluation Step
# ============================================================
# Philosophy:
#
# The evaluation step validates the trained model
# before it can move forward in the CI/CD workflow.
#
# This step runs src/evaluation.py as a SageMaker
# Processing Job.
#
# The output evaluation.json contains:
#
# - accuracy
# - precision
# - recall
# - F1-score
# - ROC-AUC
# - confusion matrix
#
# The F1-score is later used by the Condition Step
# to decide whether the model should be registered
# and used for Batch Transform.
# ============================================================

script_eval = ScriptProcessor(
    image_uri=xgboost_image_uri,
    command=["python3"],
    role=role,
    instance_type=processing_instance_type,
    instance_count=1,
    base_job_name="diabetes-evaluate",
    sagemaker_session=pipeline_session
)

eval_args = script_eval.run(
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model"
        ),
        ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/test"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/evaluation"
        )
    ],
    code="src/evaluation.py",
    arguments=[
        "--prediction-threshold", prediction_threshold.to_string()
    ]
)

evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json"
)

step_eval = ProcessingStep(
    name="DiabetesEvaluate",
    step_args=eval_args,
    property_files=[evaluation_report]
)

print("Evaluation step defined.")

Evaluation step defined.


In [19]:
# ============================================================
# 8. Define Model Creation, Registration, and Batch Transform
# ============================================================

model = Model(
    image_uri=xgboost_image_uri,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    role=role,
    sagemaker_session=pipeline_session
)

step_create_model = ModelStep(
    name="DiabetesCreateModel",
    step_args=model.create(
        instance_type="ml.m5.large"
    )
)

transformer = Transformer(
    model_name=step_create_model.properties.ModelName,
    instance_type=transform_instance_type,
    instance_count=1,
    output_path=f"s3://{BUCKET_NAME}/pipeline/batch-transform-output",
    sagemaker_session=pipeline_session
)

step_transform = TransformStep(
    name="DiabetesBatchTransform",
    transformer=transformer,
    inputs=TransformInput(
        data=step_process.properties.ProcessingOutputConfig.Outputs["batch"].S3Output.S3Uri,
        content_type="text/csv",
        split_type="Line"
    )
)

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri="{}/evaluation.json".format(
            step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]
        ),
        content_type="application/json"
    )
)

register_args = model.register(
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.m5.large"],
    transform_instances=["ml.m5.large"],
    model_package_group_name=MODEL_PACKAGE_GROUP_NAME,
    approval_status=model_approval_status,
    model_metrics=model_metrics
)

step_register = ModelStep(
    name="DiabetesRegisterModel",
    step_args=register_args
)

print("Model creation, registration, and Batch Transform steps defined.")

Model creation, registration, and Batch Transform steps defined.


In [20]:
# ============================================================
# 9. Define F1 Condition and Fail Step
# ============================================================
# Philosophy:
#
# The Condition Step is the quality gate in the
# CI/CD pipeline.
#
# It checks the F1-score from evaluation.json.
#
# If F1-score is greater than or equal to the
# threshold:
#
# - create the model
# - register the model
# - run Batch Transform
#
# If F1-score is below the threshold:
#
# - stop the pipeline
# - mark the execution as failed
#
# This demonstrates automated model release control.
# ============================================================

step_fail = FailStep(
    name="DiabetesF1Fail",
    error_message=Join(
        on=" ",
        values=[
            "Pipeline failed because F1 score was below threshold:",
            f1_threshold
        ]
    )
)

condition_f1 = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="classification_metrics.f1.value"
    ),
    right=f1_threshold
)

step_condition = ConditionStep(
    name="DiabetesF1Condition",
    conditions=[condition_f1],
    if_steps=[
        step_create_model,
        step_register,
        step_transform
    ],
    else_steps=[
        step_fail
    ]
)

print("Condition and fail steps defined.")

Condition and fail steps defined.


In [21]:
# ============================================================
# 10. Create SageMaker Pipeline Definition
# ============================================================
# Philosophy:
#
# This cell connects all pipeline steps into one
# directed acyclic graph, or DAG.
#
# The final pipeline flow is:
#
# Preprocess
#   ↓
# Train XGBoost using train.py
#   ↓
# Evaluate using evaluation.py
#   ↓
# F1 Condition
#   ├── Pass: Register Model + Batch Transform
#   └── Fail: Stop pipeline safely
#
# This is the main CI/CD automation layer for the
# final project.
# ============================================================

pipeline = Pipeline(
    name=PIPELINE_NAME,
    parameters=[
        processing_instance_count,
        processing_instance_type,
        training_instance_type,
        transform_instance_type,
        model_approval_status,
        input_data,
        f1_threshold,
        prediction_threshold,
        sample_size_param,
        production_size_param
    ],
    steps=[
        step_process,
        step_train,
        step_eval,
        step_condition
    ],
    sagemaker_session=pipeline_session
)

definition = json.loads(pipeline.definition())

print("Pipeline definition created.")
print("Pipeline name:", PIPELINE_NAME)

Pipeline definition created.
Pipeline name: DiabetesReadmissionPipeline


In [22]:
# ============================================================
# 11. Upsert SageMaker Pipeline
# ============================================================
# Philosophy:
#
# Upserting the pipeline creates it if it does not
# already exist, or updates it if the definition has
# changed.
#
# SageMaker will store this as
# the updated pipeline definition and future
# executions will use the modular script-based
# workflow.
# ============================================================

pipeline.upsert(
    role_arn=role
)

print("Pipeline upsert complete.")
print("Pipeline name:", PIPELINE_NAME)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Pipeline upsert complete.
Pipeline name: DiabetesReadmissionPipeline


In [23]:
# ============================================================
# 12. Start Successful Pipeline Execution
# ============================================================
# Philosophy:
#
# This execution uses a realistic F1 threshold.
#
# If the model achieves an F1-score greater than
# or equal to the threshold, the pipeline should:
#
# - pass the condition step
# - register the model
# - run Batch Transform
#
# This execution is used as the successful CI/CD
# example for the final project.
# ============================================================

success_execution = pipeline.start(
    parameters={
        "F1Threshold": 0.20,
        "PredictionThreshold": 0.50,
        "ModelApprovalStatus": "PendingManualApproval"
    }
)

print("Successful pipeline execution started:")
print(success_execution.arn)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Successful pipeline execution started:
arn:aws:sagemaker:us-east-1:468962265940:pipeline/DiabetesReadmissionPipeline/execution/7w9yemvhz3qm


In [26]:
# ============================================================
# 13. Monitor Pipeline Execution
# ============================================================
# Philosophy:
#
# This cell displays the current status of each
# pipeline step.
#
# During execution we can observe:
#
# - Preprocessing
# - Training
# - Evaluation
# - Condition Step
# - Model Registration
# - Batch Transform
#
# This provides visibility into the CI/CD workflow.
# ============================================================

execution_steps = success_execution.list_steps()

for step in execution_steps:
    print(step)
    print("-" * 80)

{'StepName': 'DiabetesBatchTransform', 'StartTime': datetime.datetime(2026, 6, 19, 8, 54, 22, 720000, tzinfo=tzlocal()), 'EndTime': datetime.datetime(2026, 6, 19, 8, 59, 25, 136000, tzinfo=tzlocal()), 'StepStatus': 'Succeeded', 'Metadata': {'TransformJob': {'Arn': 'arn:aws:sagemaker:us-east-1:468962265940:transform-job/pipelines-7w9yemvhz3qm-DiabetesBatchTransfo-qC2HbaonWq'}}, 'AttemptCount': 1}
--------------------------------------------------------------------------------
{'StepName': 'DiabetesCreateModel-CreateModel', 'StartTime': datetime.datetime(2026, 6, 19, 8, 54, 20, 498000, tzinfo=tzlocal()), 'EndTime': datetime.datetime(2026, 6, 19, 8, 54, 22, 226000, tzinfo=tzlocal()), 'StepStatus': 'Succeeded', 'Metadata': {'Model': {'Arn': 'arn:aws:sagemaker:us-east-1:468962265940:model/pipelines-7w9yemvhz3qm-DiabetesCreateModel--fUhdCeqRAv'}}, 'AttemptCount': 1}
--------------------------------------------------------------------------------
{'StepName': 'DiabetesRegisterModel-RegisterMo

In [27]:
from pprint import pprint
from sagemaker.s3 import S3Downloader
import json

evaluation_s3_uri = "{}/evaluation.json".format(
    step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]
)

evaluation_json = S3Downloader.read_file(evaluation_s3_uri)

evaluation_report_dict = json.loads(evaluation_json)

pprint(evaluation_report_dict)

{'classification_metrics': {'accuracy': {'value': 0.748},
                            'confusion_matrix': {'value': [[1059, 274],
                                                           [104, 63]]},
                            'f1': {'value': 0.24999999999999994},
                            'precision': {'value': 0.18694362017804153},
                            'prediction_threshold': {'value': 0.5},
                            'recall': {'value': 0.3772455089820359},
                            'roc_auc': {'value': 0.6436474388058093}}}


In [33]:
publish_pipeline_metrics(
    evaluation_report_dict=evaluation_report_dict,
    region=REGION,
    namespace=CLOUDWATCH_NAMESPACE,
    pipeline_name=PIPELINE_NAME
)

Published metrics to CloudWatch namespace:
AAI540/DiabetesReadmissionPipeline


Published metrics to CloudWatch namespace:
AAI540/DiabetesReadmissionPipeline


Published metrics to CloudWatch namespace:
AAI540/DiabetesReadmissionPipeline


Published metrics to CloudWatch namespace:
AAI540/DiabetesReadmissionPipeline


Published metrics to CloudWatch namespace:
AAI540/DiabetesReadmissionPipeline


In [34]:
create_pipeline_dashboard(
    region=REGION,
    namespace=CLOUDWATCH_NAMESPACE,
    pipeline_name=PIPELINE_NAME,
    dashboard_name=DASHBOARD_NAME
)

CloudWatch dashboard created:
AAI540-DiabetesReadmission-Pipeline-Monitoring


CloudWatch dashboard created:
AAI540-DiabetesReadmission-Pipeline-Monitoring


CloudWatch dashboard created:
AAI540-DiabetesReadmission-Pipeline-Monitoring


In [35]:
batch_output_s3_uri = f"s3://{BUCKET_NAME}/pipeline/batch-transform-output/"

print("Batch Transform output location:")
print(batch_output_s3_uri)

Batch Transform output location:
s3://aai540-diabetes-readmission-468962265940/pipeline/batch-transform-output/


Batch Transform output location:
s3://aai540-diabetes-readmission-468962265940/pipeline/batch-transform-output/


Batch Transform output location:
s3://aai540-diabetes-readmission-468962265940/pipeline/batch-transform-output/
